# How to overlay non-Roman source catalogs
***

## Learning Goals

This is a beginner tutorial covering loading non-Roman source catalogs into [MastAladin](https://github.com/spacetelescope/mast-aladin). We will cover some of the basic tools provided by astroquery to programatically query for sources and the tools within MastAladin to load the sources into the viewer. By the end of this tutorial, you will:

- Understand the basics of querying for source catalogs with astroquery tools.
- Be able to load an astroquery Table of source catalogs into `MastAladin`.
- Know how to programatically interface with mast aladin source Tables.

## Table of Contents
- [Introduction](#Introduction)
- [Imports](#Imports)
- [MastAladin Setup](#MastAladin-Setup)
- [Exercise 1: Catalogs](#Exercise-1:-Catalogs)
- [Exercise 2: MastMissions](#Exercise-2:-MastMissions)
- [Exercise 3: MastAladin UI](#Exercise-3:-MastAladin-UI)

## Introduction

This tutorial covers how to query for non-Roman source catalogs with a variety of tools provided by the [astoryquery.mast](https://astroquery.readthedocs.io/en/stable/mast/mast.html) library and how to load the results into the `MastAladin` viewer.

This tutotiral will cover the following services provided by `astroquery.mast`.

- [MastMissions](https://astroquery.readthedocs.io/en/latest/mast/mast_missions.html): allows for search queries based on mission-specific metadata for a given data collection
- [Catalogs](https://astroquery.readthedocs.io/en/stable/mast/mast_catalog.html): used to query MAST catalog data

We will also cover how to use the `MastAladin.add_table()` API to load the source catalogs into the `MastAladin` viewer.


## Imports

We will import the following in this tutorial:

- astroquery.mast: query and download source catalogs for a subset of the astronomical catalogs stored at MAST
- mast_aladin.MastAladin: interactive sky map widget for visualizing source catalogs
- sidecar: organize our MastAladin viewer widget

In [ ]:
from astroquery.mast import Catalogs, MastMissions
from mast_aladin import MastAladin
from sidecar import Sidecar

***
### MastAladin Setup
The first step to working with `MastAladin` is to setup our notebook workspace. We start by initializing a new instance of `MastAladin` with the `target` set to `M31` and the `fov` set to  `10` to ensure we can see the source catalogs that we will be loading.

In [ ]:
aladin = MastAladin(target="M31", fov=10)

we then utilize the `Sidecar` library to display the viewer in a separate window. More information on this can be found in the [interactive_data_exploration/how_to/organize_widgets/organize_widgets.ipynb](../organize_widgets/organize_widgets.ipynb) tutorial.

In [ ]:
sc = Sidecar(title="mast-aladin", anchor="split-right")
# Display the widget within the sidecar
with sc:
    display(aladin)

***
### Exercise 1: Catalogs

The `astroyquery.mast.Catalogs` class provides access to a subset of the astronomical catalogs stored at MAST. For this example, we will look at the `PanSTARRS DR2` source catalog. The full list of catalogs currently available through this interface are available to view [here](https://astroquery.readthedocs.io/en/stable/mast/mast_catalog.html).

#### step 1: query_object

For this exercise, we will query for PanSTARRS DR2 source catalog objects near the core of M31 using the [query_object](https://astroquery.readthedocs.io/en/stable/api/astroquery.mast.CatalogsClass.html#astroquery.mast.CatalogsClass.query_object) method. PanSTARRs contains a large number of sources, so we also include a radius of `.02` to ensure the query runtime is reasonable. 

In [ ]:
panstarrs_catalog = Catalogs.query_object("m31", radius=.02, catalog="PANSTARRS", data_release="dr2")

#### step 2: add_table
The result of this query is an [astropy Table](https://docs.astropy.org/en/stable/table/index.html) which can be readily loaded into `MastAladin`

In [ ]:
aladin.add_table(panstarrs_catalog)

But wait, if you look at the `MastAladin` viewer on the right, it seems like nothing has been loaded! Zooming out in the viewer will show that the table was loaded, but the sources were place incorrectly on the skymap. 

![alt text](panstarrs_loaded_with_default_ra_dec.png)

This is because `MastAladin` tries to assume the `ra` and `dec` fields automatically, but is not always correct. To fix this, specify the correct `ra` and `dec` fields as found in the PanSTARRS source catalog table. Optionally, a custom name can also be added with the `name` parameter.

In [ ]:
aladin.add_table(panstarrs_catalog, name="panstarrs", raField="raMean", decField="decMean")

After running the above command, the PanSTARRS source catalogs should populate around the core of M31

![alt-text](panstarrs_correct_ra_dec.png)

#### step 4: remove_overlay

Unused catalogs loaded into aladin can cause performance issues. As a final step to this exercise we will show how to use the `remove_overlay` method to remove catalogs. 

First, identify the source catalogs to remove. Once loaded into `MastAladin`, catalogs are identified as an `overlay` and their IDs can be listed with the `aladin.overlays` command.

In [ ]:
aladin.overlays

Next, provide the ID of the source catalog to the `remove_overlay` method. The ID of the catalog to remove should be `catalog` in this instance.

In [ ]:
aladin.remove_overlay('catalog')

***
### Exercise 2: MastMissions
MissionsMast provides direct programmatic access to the MAST portal for querying Mast missions such as Roman, JWST, etc. This example will cover querying for JWST source catalogs.

#### Step 1: Initialize MastMissions
The MastMission class must be initialize with the mission to query from, in this case that would be `jwst`.

In [ ]:
mm = MastMissions(mission="jwst")

#### Step 2: query_region
There are multiple supported ways to make a query in astroquery. This example covers the [query_region](https://astroquery.readthedocs.io/en/latest/api/astroquery.mast.MastMissionsClass.html#astroquery.mast.MastMissionsClass.query_region) method, however the `query_object` method from Example 1 would still work. We also require a much larger radius to get any results back for JWST.

In [ ]:
jwst_catalog = mm.query_region(
    coordinates="10.6847083 +41.2687500",
    radius=30
)

#### Step 3: add_table
Same as with the PanSTARRS example, provide an explicit `ra` and `dec` when adding the table of source catalogs from JWST.

In [ ]:
jwst_table = aladin.add_table(jwst_catalog, name="jwst", raField="targ_ra", decField="targ_dec")

![alt-text](jwst_source_catalog_loaded.png)

### Exercise 3: MastAladin UI
The MastAladin UI provides the ability to load source catalogs for `GAIA DR3` and `2MASS` through the UI. The sources will be loaded based on the current viewer position and FOV.


![alt-text](gaia_steps_to_load_source_catalog.png)

#### Steps:
1. Click the stacks icon
2. Click add overlay icon
3. Select Catalogs
4. Select GAIA DR3

![alt-text](gaia_dr3_source_catalog_loaded.png)



## About this Notebook

**Author(s):** Patrick Custer <br>
**Keyword(s):** mast-aladin, astroquery, source catalogs <br>
**First published:** July 2026 <br>
**Last updated:** July 2026 <br>

***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 